# Cost এবং Latency Optimization

তিনটি demo:

1. **Batching trade-off:** এলোমেলোভাবে আসা request-এর একটি stream একটি single simulated server-এ size B-এর batch-এ ভাগ করা হয়; B-এর একটি sweep জুড়ে throughput এবং গড় per-request latency **দুটোই** মাপা হয়, সেই ক্ষেত্রসহ যেখানে batch size এত ছোট যে arrival rate-এর সাথেও তাল রাখতে পারে না (queue backlog)।
2. **একটি খেলনা model-routing cascade:** একটি সস্তা আগাম "difficulty" classifier সহজ query-গুলো একটি সস্তা/দ্রুত model-এ এবং কঠিন query-গুলো একটি ব্যয়বহুল/নির্ভুল model-এ পাঠায়, আসল মাপা cost ও accuracy-তে always-cheap এবং always-expensive baseline-এর সাথে তুলনা করা হয়।
3. **GPU-hour economics:** কয়েকটি দৃষ্টান্তমূলক (GPU rental cost, মাপা tokens/sec) configuration-কে একটি আসল $/million-tokens সংখ্যায় রূপান্তর, দেখানো যে quantization ও model-sizing সিদ্ধান্ত শুধু throughput/memory lever নয়, মূর্ত cost lever-ও।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। তিনটি demo একই global `rng` ভাগ করে, তাই ক্রমানুসারে চালালে output মূল script-এর সাথে হুবহু মিলবে।

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## 1. Batching trade-off: throughput বনাম latency

`simulate_batching()` একটি single-server batching simulation: request একটি একটি করে আসে; `batch_size`-টি request queue-তে জমা হলেই একটি batch তৈরি হয় এবং একক ইউনিট হিসেবে process হয় (স্থির overhead + প্রতি request-এ সামান্য অতিরিক্ত সময়)। Demo-টি বিভিন্ন B-এর জন্য throughput, গড় latency ও সর্বোচ্চ latency-র table দেখায় — কোনো একক batch size দুটোই একসাথে কমাতে পারে না।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Batching trade-off: throughput বনাম latency
# ---------------------------------------------------------------------------

def simulate_batching(batch_size, num_requests=600, mean_interarrival_ms=10.0,
                       overhead_ms=50.0, marginal_ms_per_request=2.0):
    """একটি single-server batching simulation। Request একটি একটি করে আসে;
    `batch_size`-টি request queue-তে জমা হওয়া মাত্র একটি batch তৈরি হয়, তারপর
    একক ইউনিট হিসেবে process হয় (স্থির overhead + প্রতি request-এ সামান্য
    সংযোজন, আসল hardware-এর সাথে মিল রেখে যেখানে একটি batch আইটেমগুলো
    একটি একটি করে চালানোর চেয়ে per-item অনেক সস্তা, কিন্তু scaling cost থেকে
    সম্পূর্ণ মুক্ত নয়)। Server একবারে একটি batch process করে (একটি single
    simulated GPU)।"""
    interarrival_times = rng.exponential(mean_interarrival_ms, size=num_requests)
    arrival_times = np.cumsum(interarrival_times)

    processing_time = overhead_ms + marginal_ms_per_request * batch_size

    latencies = []
    server_free_at = 0.0
    for start in range(0, num_requests - num_requests % batch_size, batch_size):
        chunk = arrival_times[start:start + batch_size]
        fill_time = chunk[-1]                                  # batch কখন পূর্ণ হয়
        batch_start = max(fill_time, server_free_at)             # server তখনো ব্যস্ত থাকতে পারে
        batch_end = batch_start + processing_time
        server_free_at = batch_end
        latencies.extend(batch_end - chunk)                      # batch-এর প্রতিটি request একসাথে শেষ হয়

    latencies = np.array(latencies)
    total_time_ms = server_free_at - arrival_times[0]
    throughput_per_sec = (len(latencies) / total_time_ms) * 1000.0
    return throughput_per_sec, latencies.mean(), latencies.max()


def batching_tradeoff_demo():
    print("=" * 70)
    print("1. THE BATCHING TRADE-OFF: THROUGHPUT vs. LATENCY")
    print("=" * 70)
    mean_interarrival_ms = 10.0
    arrival_rate_per_sec = 1000.0 / mean_interarrival_ms
    print(f"Requests arrive with mean interarrival {mean_interarrival_ms:.0f}ms "
          f"(~{arrival_rate_per_sec:.0f} requests/sec on average).")
    print("Batch cost model: 50ms fixed overhead + 2ms per additional request in the batch.\n")

    print(f"{'batch size':>10}{'throughput/sec':>16}{'avg latency (ms)':>20}{'max latency (ms)':>20}")
    for B in [1, 2, 4, 8, 16, 32, 64]:
        throughput, avg_latency, max_latency = simulate_batching(B)
        print(f"{B:>10}{throughput:>16.1f}{avg_latency:>20.1f}{max_latency:>20.1f}")

    print("\n-> At B=1, each request pays the full 52ms processing cost alone, but")
    print("   requests arrive every ~10ms on average -- the server cannot keep up,")
    print("   a backlog builds across the whole run, and both average AND max")
    print("   latency blow up far beyond what a single request's own processing")
    print("   time would suggest. Growing B raises the throughput ceiling (more")
    print("   requests completed per unit of server-busy-time) enough to escape")
    print("   that backlog, but every request in a bigger batch waits longer for")
    print("   the batch to FILL before processing even starts -- so latency keeps")
    print("   climbing even once throughput is no longer the bottleneck. There is")
    print("   no single batch size that minimizes both at once.")


batching_tradeoff_demo()

## 2. Model routing / cascade

সস্তা model-এর accuracy query কঠিন হওয়ার সাথে কমে যায়, আর ব্যয়বহুল model পুরো difficulty পরিসরে উঁচু accuracy বজায় রাখে (দাম 20 গুণ)। `train_difficulty_router()` একটি noisy, সস্তায় গণনাযোগ্য difficulty proxy-র উপর সেই routing threshold বেছে নেয় যা (accuracy) - (cost / scale) সর্বোচ্চ করে। Demo-টি test set-এ always-cheap, always-expensive ও cascade-এর গড় cost ও accuracy তুলনা করে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Model routing / cascade
# ---------------------------------------------------------------------------

COST_CHEAP = 1.0
COST_EXPENSIVE = 20.0


def cheap_model_is_correct(difficulty):
    """Query যত কঠিন হয়, সস্তা model-এর accuracy তত কমে যায়।"""
    p_correct = np.clip(0.97 - 0.7 * difficulty, 0.05, 0.97)
    return rng.random(size=difficulty.shape) < p_correct


def expensive_model_is_correct(difficulty):
    """ব্যয়বহুল model পুরো difficulty পরিসর জুড়েই উচ্চ accuracy বজায় রাখে।"""
    p_correct = np.clip(0.97 - 0.08 * difficulty, 0.05, 0.97)
    return rng.random(size=difficulty.shape) < p_correct


def train_difficulty_router(threshold_candidates, n_train=2000):
    """Difficulty-র একটি noisy, সস্তায় গণনাযোগ্য proxy feature-এর উপর সেই
    routing threshold বেছে নেয় যা একটি held-out validation set-এ (অর্জিত
    accuracy) - (প্রদত্ত cost / scale) সর্বোচ্চ করে -- "escalation সার্থক কিনা
    তা predict করতে train করা একটি ছোট classifier"-এর একটি সরল, সৎ পরিবর্ত।"""
    true_difficulty = rng.uniform(0, 1, size=n_train)
    observed_feature = np.clip(true_difficulty + rng.normal(0, 0.15, size=n_train), 0, 1)

    best_threshold, best_score = None, -np.inf
    for t in threshold_candidates:
        route_expensive = observed_feature > t
        correct = np.where(
            route_expensive,
            expensive_model_is_correct(true_difficulty),
            cheap_model_is_correct(true_difficulty),
        )
        cost = np.where(route_expensive, COST_EXPENSIVE, COST_CHEAP)
        # Accuracy-কে পুরস্কৃত, cost-কে দণ্ডিত করা -- যে trade-off একটি আসল deployment-কে বেছে নিতে হয়।
        score = correct.mean() - 0.02 * cost.mean()
        if score > best_score:
            best_score, best_threshold = score, t
    return best_threshold


def cascade_demo():
    print("\n" + "=" * 70)
    print("2. MODEL ROUTING / CASCADES: CHEAP-BY-DEFAULT, ESCALATE WHEN NEEDED")
    print("=" * 70)

    threshold = train_difficulty_router(np.linspace(0.05, 0.95, 19))
    print(f"Learned routing threshold on the (noisy) difficulty proxy: {threshold:.2f}")
    print("(observed_feature > threshold -> escalate to the expensive model)\n")

    n_test = 5000
    true_difficulty = rng.uniform(0, 1, size=n_test)
    observed_feature = np.clip(true_difficulty + rng.normal(0, 0.15, size=n_test), 0, 1)

    always_cheap_correct = cheap_model_is_correct(true_difficulty)
    always_expensive_correct = expensive_model_is_correct(true_difficulty)

    route_expensive = observed_feature > threshold
    cascade_correct = np.where(
        route_expensive,
        expensive_model_is_correct(true_difficulty),
        cheap_model_is_correct(true_difficulty),
    )
    cascade_cost = np.where(route_expensive, COST_EXPENSIVE, COST_CHEAP)

    print(f"{'strategy':>18}{'avg cost/query':>18}{'accuracy':>12}")
    print(f"{'always cheap':>18}{COST_CHEAP:>18.2f}{always_cheap_correct.mean():>12.3f}")
    print(f"{'always expensive':>18}{COST_EXPENSIVE:>18.2f}{always_expensive_correct.mean():>12.3f}")
    print(f"{'cascade':>18}{cascade_cost.mean():>18.2f}{cascade_correct.mean():>12.3f}")

    escalated_fraction = route_expensive.mean()
    cost_saved_vs_expensive = 1 - cascade_cost.mean() / COST_EXPENSIVE
    accuracy_gap_vs_expensive = always_expensive_correct.mean() - cascade_correct.mean()

    midpoint_cost = (COST_CHEAP + COST_EXPENSIVE) / 2
    midpoint_accuracy = (always_cheap_correct.mean() + always_expensive_correct.mean()) / 2
    print(f"\nEscalated to the expensive model: {escalated_fraction:.1%} of queries")
    print(f"Cost saved vs. always using the expensive model: {cost_saved_vs_expensive:.1%}")
    print(f"Accuracy gap vs. always using the expensive model: {accuracy_gap_vs_expensive:.3f}")
    print(f"\nFor reference, simply averaging the two baselines 50/50 would cost "
          f"{midpoint_cost:.2f} at {midpoint_accuracy:.3f} accuracy.")
    print(f"The cascade's cost ({cascade_cost.mean():.2f}) landed close to that same")
    print(f"midpoint, but its accuracy ({cascade_correct.mean():.3f}) came in ABOVE the")
    print(f"midpoint -- because it isn't routing randomly, it's deliberately sending the")
    print(f"queries where the cheap model is weakest up to the expensive model, so the")
    print(f"same escalation budget buys more accuracy than a coin flip would.")
    print("   The real risk (per the README) is the escalation decision being wrong on")
    print("   genuinely hard queries the cheap proxy feature underestimates -- that")
    print("   shows up ONLY in the accuracy gap above, never in the cost numbers, which")
    print("   is exactly why a cascade needs both metrics evaluated before shipping,")
    print("   not cost alone.")


cascade_demo()

## 3. GPU-hour economics: throughput + rental cost -> $/million tokens

মূল cost-engineering সূত্র (README section 4): একটি GPU-এর ঘণ্টাপ্রতি খরচ, এক ঘণ্টায় এটি আসলে যত token তৈরি করে তার উপর ভাগ করা। Demo-টি কয়েকটি দৃষ্টান্তমূলক configuration (7B fp16, 7B INT8, 13B fp16, 70B 8-GPU node)-কে $/1M tokens-এ রূপান্তর করে এবং দেখায় quantization ও model sizing কীভাবে মূর্ত cost lever।

In [ ]:
# ---------------------------------------------------------------------------
# 3. GPU-hour economics: throughput + rental cost -> $/million tokens
# ---------------------------------------------------------------------------

def cost_per_million_tokens(gpu_cost_per_hour, tokens_per_sec):
    """মূল cost-engineering সূত্র (README section 4): একটি GPU-এর ঘণ্টাপ্রতি
    খরচ, এক ঘণ্টায় এটি আসলে যত token তৈরি করে তার উপর ছড়িয়ে দেওয়া।"""
    tokens_per_hour = tokens_per_sec * 3600.0
    return gpu_cost_per_hour / tokens_per_hour * 1_000_000.0


def gpu_economics_demo():
    print("\n" + "=" * 70)
    print("3. GPU-HOUR ECONOMICS: FROM THROUGHPUT TO $/MILLION TOKENS")
    print("=" * 70)
    print("Illustrative configurations (NOT specific real vendor pricing/benchmarks --")
    print("the point is the FORMULA and the relative comparison, not these exact numbers):\n")

    # (label, gpu_cost_per_hour $, মাপা সমষ্টিগত tokens/sec)
    configs = [
        ("7B, fp16, single GPU",            3.00,  120.0),
        ("7B, INT8 quantized, single GPU",  3.00,  210.0),   # Lesson 2: ছোট weight -> একই GPU-তে বেশি throughput
        ("13B, fp16, single GPU",           3.00,   70.0),
        ("70B, fp16, 8-GPU node",          24.00,  450.0),   # সমষ্টিগত node throughput, 8টি GPU জুড়ে tensor-parallel
    ]

    print(f"{'configuration':>32}{'$/GPU-hr':>12}{'tokens/sec':>14}{'$/1M tokens':>16}")
    costs = {}
    for label, gpu_cost, tps in configs:
        cost = cost_per_million_tokens(gpu_cost, tps)
        costs[label] = cost
        print(f"{label:>32}{gpu_cost:>12.2f}{tps:>14.1f}{cost:>16.3f}")

    fp16_cost = costs["7B, fp16, single GPU"]
    int8_cost = costs["7B, INT8 quantized, single GPU"]
    big_cost = costs["70B, fp16, 8-GPU node"]
    savings = (1 - int8_cost / fp16_cost) * 100
    print(f"\n-> Quantizing the SAME 7B model to INT8 (Lesson 2) raises tokens/sec on the SAME")
    print(f"   single GPU (memory-bandwidth-bound decode reads fewer bytes per step, Lesson 1")
    print(f"   section 6) with no change in GPU rental cost -- cutting $/million tokens by")
    print(f"   {savings:.0f}% (from {fp16_cost:.3f} to {int8_cost:.3f}). This is quantization's")
    print("   memory/throughput win, converted into an actual dollar figure.")
    print(f"\n-> The 70B model's node costs {big_cost:.3f} $/million tokens -- "
          f"{big_cost/int8_cost:.1f}x the quantized 7B's cost per token. Model SIZING is a")
    print("   real cost decision: paying for a bigger model only makes sense for the traffic")
    print("   that actually needs its extra quality -- exactly what section 3's cascade")
    print("   routes to it selectively, instead of sending every query through it by default.")


gpu_economics_demo()

## সবগুলো demo একসাথে

`main()` তিনটি demo — batching trade-off, model-routing cascade এবং GPU-hour economics — এক চালে পরপর চালায়, ঠিক মূল script-এর মতো। উপরের cell-গুলোতে demo-গুলো ইতিমধ্যে চলেছে, তাই দুবার না চালানোর জন্য call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    batching_tradeoff_demo()
    cascade_demo()
    gpu_economics_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন